# Prior-Driven Drift Diagnostics

Verify that the score-based drift (`x ← x + ε · score(x)`) is working
correctly by checking expected signatures:

1. **Raw score norm decreases** — points approach the mode where ∇ log p is small.
2. **L2 to nearest clean decreases** — drift pulls noisy points back.
3. **Median k-NN L2 distance decreases** — more robust than single-nearest (less sensitive to pool density).
4. **Median k-NN cosine distance decreases** — directional alignment with the data manifold, robust via k-NN.
5. **Cosine distance to source decreases** — drift recovers the *correct* stimulus, not just any nearby one.
6. **Log-likelihood increases** — histogram of Δlog p shifts rightward over time.

Uses `utls.drift_diagnostics` which tracks all metrics plus cumulative
Δlog p via the first-order Taylor expansion: Δlog p ≈ score(x) · Δx.

In [ ]:
import sys, os, yaml, torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from utls.runners_utils import *
from utls.loading import (
    load_results_with_exclusion_2,
    move_sequences_to_used,
    load_results_with_exclusion_no_dropping,
)
from encoders import *
from ScoreFunction import ScoreFunction
from utls.drift_diagnostics import (
    drift_trajectory,
    plot_drift_diagnostic,
    drift_diagnostic_batch,
    plot_loglik_histograms,
)

## 1. Load config, encode stimuli, load score model

In [ ]:
CONFIG_PATH = (
    "/om2/user/bjmedina/auditory-memory/memory/"
    "model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml"
)

with open(CONFIG_PATH) as f:
    model_cfg = yaml.safe_load(f)

exp_cfg  = model_cfg["experiment"]
repr_cfg = model_cfg["representation"]

which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
print(f"Stimuli: {len(all_files)},  Sequences: {len(exp_list)}")

In [ ]:
encoder_type = repr_cfg["type"]
layer   = repr_cfg.get("layer", None)
pc_dims = repr_cfg.get("pc_dims", None)

NN_ENCODERS = {"kell2018", "resnet50"}
encoder_task = "word_speaker_audioset" if encoder_type in NN_ENCODERS else "audioset"

encoder_cfg = dict(
    encoder_type=encoder_type,
    model_name=encoder_type,
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=repr_cfg["time_avg"],
    device="cuda",
)
if encoder_type in NN_ENCODERS:
    encoder_cfg["layer"] = layer

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)
print(f"Encoded shape: {X.shape}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

score_fn = ScoreFunction(
    mode="textures",
    restart=True,
    likelihood_eval=True,
    device=DEVICE,
)
print(f"Score model loaded on {DEVICE}")

# Quick sanity check: forward pass on one point
x_test = X[0:1].to(DEVICE)
unit_out = score_fn.forward(x_test)
raw_out  = score_fn.forward_raw(x_test)
print(f"Unit-norm output shape: {unit_out.shape}, norm: {unit_out.reshape(-1).norm():.4f}")
print(f"Raw output shape: {raw_out.shape}, norm: {raw_out.reshape(-1).norm():.4f}")

---
## 2. Single-point drift trajectory

Pick a random clean encoding, corrupt it with noise, then drift it
back toward the prior.  Track:
- **Row 1:** Raw score norm, L2 to nearest, step displacement
- **Row 2:** Median k-NN L2, median k-NN cosine dist, cosine dist to source

In [ ]:
NOISE_STD  = 0.3   # amount of corruption
STEP_SIZE  = 0.001  # drift step size (match your runner's drift_step_size)
N_STEPS    = 300    # drift iterations
KNN_K      = 5      # neighbours for median k-NN distance

# Pick a random clean point and add noise
rng = np.random.default_rng(42)
idx = rng.integers(len(X))
x_clean = X[idx].to(DEVICE)
x_noisy = x_clean + NOISE_STD * torch.randn_like(x_clean)

print(f"Clean point idx={idx}, L2 norm: {x_clean.norm():.2f}")
print(f"Noise added: std={NOISE_STD}, displacement: {(x_noisy - x_clean).norm():.4f}")

traj = drift_trajectory(
    score_model=score_fn,
    x_start=x_noisy,
    step_size=STEP_SIZE,
    n_steps=N_STEPS,
    X_clean=X.to(DEVICE),
    x_source=x_clean,       # track cosine dist back to the original stimulus
    use_unit_norm=True,
    knn_k=KNN_K,
)

plot_drift_diagnostic(traj, title="Single-point drift trajectory");

---
## 3. Batch diagnostic (mean ± std across many starting points)

Run drift from 20 randomly corrupted clean points and plot
aggregate stats across all distance metrics (L2, k-NN, cosine)
plus cosine distance to source.

In [ ]:
trajs = drift_diagnostic_batch(
    score_model=score_fn,
    X_clean=X.to(DEVICE),
    n_samples=20,
    noise_std=NOISE_STD,
    step_size=STEP_SIZE,
    n_steps=N_STEPS,
    seed=42,
    knn_k=KNN_K,
)

---
## 4. Log-likelihood histogram "movie"

Take ~1000 noisy points, drift them, and show the distribution of
cumulative Δlog p(x) at t=0, t=T/2, and t=T.  The histograms
should shift rightward — meaning drift is pushing points uphill
on the log-density surface.

In [ ]:
fig, delta_logp = plot_loglik_histograms(
    score_model=score_fn,
    X_clean=X.to(DEVICE),
    n_samples=1000,
    noise_std=NOISE_STD,
    step_size=STEP_SIZE,
    n_steps=N_STEPS,
    n_snapshots=3,       # t=0, T/2, T
    seed=42,
    batch_size=64,
)

In [ ]:
# Summary stats for the delta log-p at each snapshot
snap_steps = np.linspace(0, N_STEPS, 3, dtype=int)
print(f"{'Step':>6}  {'Mean':>10}  {'Median':>10}  {'Std':>10}  {'% positive':>12}")
print("-" * 55)
for t in snap_steps:
    vals = delta_logp[:, t]
    print(f"{t:6d}  {np.mean(vals):10.4f}  {np.median(vals):10.4f}  "
          f"{np.std(vals):10.4f}  {100 * np.mean(vals > 0):11.1f}%")

---
## 5. Step-size sensitivity

Compare a few drift step sizes to see how the rate of log-likelihood
increase and distance decrease depend on the step magnitude.

In [ ]:
step_sizes = [0.0005, 0.001, 0.005, 0.01]
N_STEPS_SWEEP = 200
N_SWEEP_SAMPLES = 50

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ss in step_sizes:
    sweep_trajs = []
    rng = np.random.default_rng(42)
    idxs = rng.choice(len(X), size=N_SWEEP_SAMPLES, replace=False)
    
    for i in idxs:
        x_noisy = X[i].to(DEVICE) + NOISE_STD * torch.randn(X.shape[1], device=DEVICE)
        t = drift_trajectory(
            score_model=score_fn,
            x_start=x_noisy,
            step_size=ss,
            n_steps=N_STEPS_SWEEP,
            X_clean=X.to(DEVICE),
        )
        sweep_trajs.append(t)
    
    # Aggregate
    norms = np.stack([t["raw_score_norms"] for t in sweep_trajs])
    dists = np.stack([t["dist_to_clean"] for t in sweep_trajs])
    
    steps = np.arange(N_STEPS_SWEEP)
    axes[0].plot(steps, norms.mean(0), label=f"step={ss}")
    axes[1].plot(np.arange(N_STEPS_SWEEP + 1), dists.mean(0), label=f"step={ss}")

axes[0].set_xlabel("Drift step")
axes[0].set_ylabel("||\u2207 log p(x)||")
axes[0].set_title("Raw score norm (mean)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.25)

axes[1].set_xlabel("Drift step")
axes[1].set_ylabel("L2 dist to nearest clean")
axes[1].set_title("Distance to data (mean)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.25)

fig.suptitle(f"Step-size comparison (noise_std={NOISE_STD}, n={N_SWEEP_SAMPLES})",
             y=1.03, fontsize=13)
fig.tight_layout()
plt.show()

---
## 6. Raw score norm: clean vs noisy points

If the score model learned the prior well, clean data points
should have **smaller** raw score norms (closer to mode) than
randomly corrupted versions.

In [ ]:
N_COMPARE = 200
rng = np.random.default_rng(0)
compare_idx = rng.choice(len(X), size=N_COMPARE, replace=False)

x_clean_batch = X[compare_idx].to(DEVICE)
x_noisy_batch = x_clean_batch + NOISE_STD * torch.randn_like(x_clean_batch)

# Raw score norms
raw_clean = score_fn.forward_raw(x_clean_batch).reshape(N_COMPARE, -1)
raw_noisy = score_fn.forward_raw(x_noisy_batch).reshape(N_COMPARE, -1)

norms_clean = raw_clean.norm(dim=1).cpu().numpy()
norms_noisy = raw_noisy.norm(dim=1).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(
    min(norms_clean.min(), norms_noisy.min()),
    max(norms_clean.max(), norms_noisy.max()),
    40,
)
ax.hist(norms_clean, bins=bins, alpha=0.6, label=f"Clean (median={np.median(norms_clean):.2f})")
ax.hist(norms_noisy, bins=bins, alpha=0.6, label=f"Noisy (median={np.median(norms_noisy):.2f})")
ax.set_xlabel("||\u2207 log p(x)||")
ax.set_ylabel("Count")
ax.set_title(f"Raw score norm: clean vs corrupted (noise_std={NOISE_STD})")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Clean  — mean: {norms_clean.mean():.2f}, median: {np.median(norms_clean):.2f}")
print(f"Noisy  — mean: {norms_noisy.mean():.2f}, median: {np.median(norms_noisy):.2f}")
print(f"Noisy/Clean ratio: {np.median(norms_noisy) / np.median(norms_clean):.2f}x")